In [ ]:
# %pip install sncosmo
# %pip install sfdmap2
# %pip install lightcurvelynx

In [2]:
import matplotlib.pyplot as plt
import lsdb
import numpy as np
from lightcurvelynx.astro_utils.mag_flux import flux2mag
from astropy.cosmology import Planck18
from astropy.coordinates import Distance
import astropy.units as u
from joblib import Parallel, delayed
import pandas as pd
from nested_pandas import read_parquet

In [3]:
import os
os.environ["SFD_DIR"] = "/home/m/mdai8/mdai/sfd/sfddata-master"

In [4]:
sn = lsdb.open_catalog("sncandid_w_bazin")

In [9]:
sn = sn.query("bazin_fit_reduced_chi2>0.5 and bazin_fit_reduced_chi2 < 5.0")

In [5]:
sn = sn.compute()

In [6]:
sn

In [ ]:
from lcfit import fit_single_lc  

def fit_single_lc_w_cond(lc,
                         bounds={"x1": (-4,4),
                                 "c": (-0.4,0.4),},
                         phase_range=(-10,40),
                         modelcov=False):
    return fit_single_lc(lc,mpbounds=bounds,phase_range=phase_range,modelcov=modelcov)

In [7]:
def infer_z_from_peakmag(row):
    peakflux = np.max(row)
    peakmag = flux2mag(peakflux)
    z = Distance(distmod = peakmag + 19).compute_z(cosmology=Planck18).value
    return z

In [8]:
sn = sn.map_rows(infer_z_from_peakmag, columns=["diaSource_dia_object_lc.psfFlux"], row_container="args",output_names=["z_est"],
                 append_columns=True)

In [9]:
res = fit_single_lc_w_cond(sn.iloc[1])

In [10]:
res

success      1.000000e+00
ncall        6.300000e+01
                 ...     
id           7.642663e+17
fit_error             NaN
Length: 54, dtype: float64

In [11]:
sn_to_fit = sn

In [12]:
results = Parallel(n_jobs=10)(delayed(fit_single_lc_w_cond)(row) for _index, row in sn_to_fit.iterrows())
result_df = pd.DataFrame(results)

In [13]:
result_df.to_parquet("saltfit.parquet")

In [78]:
result_df = read_parquet("saltfit.parquet")

In [79]:
result_good = result_df.loc[result_df.success == 1]

In [82]:
result_good = result_good.loc[(np.absolute(result_good.x1) < 3.9) & (np.absolute(result_good.c) < 0.39)]

In [83]:
result_good["reduced_chisq"] = result_good.chisq / result_good.ndof

In [85]:
result_good.reduced_chisq.hist(bins = np.linspace(0, 20, 20))

In [72]:
result_good = result_good.loc[(result_good.reduced_chisq > 0.1) & (result_good.reduced_chisq < 5.)]

In [73]:
sn_good = sn.iloc[result_good.index]

In [74]:
sn_good

diaObjectId_dia_object_lc  ra_dia_object_lc  \
_healpix_29                                                        
1297823073888117834         767427573947826228        335.931056   
1299065760871892174         770678073916915729        340.606183   
1299200736341842837         772302327469047813        339.583585   
1299793857969793352         770655465209069609        335.850864   
1300684541233442387         772291607230676999        337.713187   
1300852239080235114         772299578689978378         338.10538   
1300948273701518773         773939019246469138        338.773278   
1300956023687580147         773939156685422646        338.537016   
1301340138132783181         773933178090946622        338.131629   
1302626881764370315         775608215336386996        343.480496   
1303960180570224671         773948158936875140        339.762462   
1304355792897970422         773941768025538620        338.836261   

                     dec_dia_object_lc  \
_healpix_29                              
1297823073888117834         -16.587983   
1299065760871892174         -13.547314   
1299200736341842837         -13.194738   
1299793857969793352         -14.107699   
1300684541233442387         -12.317621   
1300852239080235114         -12.202653   
1300948273701518773         -11.417111   
1300956023687580147         -11.334047   
1301340138132783181         -11.050283   
1302626881764370315         -10.129434   
1303960180570224671         -10.861978   
1304355792897970422         -10.720038   

                                   diaObjectForcedSource_dia_object_lc  \
_healpix_29                                                              
1297823073888117834  [{midpointMjdTai: 60847.294432, band: 'i', psf...   
1299065760871892174  [{midpointMjdTai: 60857.340952, band: 'r', psf...   
1299200736341842837  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1299793857969793352  [{midpointMjdTai: 60847.294432, band: 'i', psf...   
1300684541233442387  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1300852239080235114  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1300948273701518773  [{midpointMjdTai: 60857.340466, band: 'r', psf...   
1300956023687580147  [{midpointMjdTai: 60857.340466, band: 'r', psf...   
1301340138132783181  [{midpointMjdTai: 60857.339965, band: 'r', psf...   
1302626881764370315  [{midpointMjdTai: 60859.380132, band: 'i', psf...   
1303960180570224671  [{midpointMjdTai: 60857.340466, band: 'r', psf...   
1304355792897970422  [{midpointMjdTai: 60857.340466, band: 'r', psf...   

                                               diaSource_dia_object_lc  \
_healpix_29                                                              
1297823073888117834  [{midpointMjdTai: 60847.319094, band: 'z', psf...   
1299065760871892174  [{midpointMjdTai: 60857.340952, band: 'r', psf...   
1299200736341842837  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1299793857969793352  [{midpointMjdTai: 60847.294432, band: 'i', psf...   
1300684541233442387  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1300852239080235114  [{midpointMjdTai: 60857.339464, band: 'r', psf...   
1300948273701518773  [{midpointMjdTai: 60857.340466, band: 'r', psf...   
1300956023687580147  [{midpointMjdTai: 60858.262977, band: 'i', psf...   
1301340138132783181  [{midpointMjdTai: 60858.262977, band: 'i', psf...   
1302626881764370315  [{midpointMjdTai: 60900.375942, band: 'i', psf...   
1303960180570224671  [{midpointMjdTai: 60860.295412, band: 'z', psf...   
1304355792897970422  [{midpointMjdTai: 60858.332845, band: 'i', psf...   

                     refExtendedness_object_lc  refSizeExtendedness_object_lc  \
_healpix_29                                                                     
1297823073888117834                        1.0                            1.0   
1299065760871892174                        1.0                       0.999993   
1299200736341842837                        1.0                            1.0   


In [7]:
def plot_lc(lc):
    for f in lc["band"].unique():
        lc_f = lc.loc[lc["band"]==f]
        is_det = np.absolute(lc_f["snr"]) > 5
        lc_det = lc_f.loc[is_det]
        lc_nondet = lc_f.loc[~is_det]
        plt.errorbar(lc_det["mjd"],lc_det["flux"],yerr=lc_det["fluxerr"],fmt='o',label=f)
        plt.errorbar(lc_nondet["mjd"],lc_nondet["flux"],yerr=lc_nondet["fluxerr"],fmt='>',alpha=0.3)

        plt.legend()
    # plt.ylim((-10000,None))

In [10]:
for i in np.random.randint(0,len(sn),size=np.min([20,len(sn)])):
    plt.subplot(2,1,1)
    lc = sn.iloc[i]["diaObjectForcedSource_dia_object_lc"]
    lc = lc[["midpointMjdTai","band","psfDiffFlux","psfDiffFluxErr"]]
    lc["mjd"] = lc["midpointMjdTai"]
    lc["flux"] = lc["psfDiffFlux"]
    lc["fluxerr"] = lc["psfDiffFluxErr"]
    lc["snr"] = lc["flux"]/lc["fluxerr"]
    plot_lc(lc)
    plt.subplot(2,1,2)
    lc = sn.iloc[i]["diaSource_dia_object_lc"]
    lc = lc[["midpointMjdTai","band","psfFlux","psfFluxErr"]]
    lc["mjd"] = lc["midpointMjdTai"]
    lc["flux"] = lc["psfFlux"]
    lc["fluxerr"] = lc["psfFluxErr"]
    lc["snr"] = lc["flux"]/lc["fluxerr"]
    plot_lc(lc)
    plt.show()